# Research09: Filtered Masking Diffusion Filtering Strength Analysis

본 노트북은 Filtered Masking Diffusion의 2단계 파라미터 분석으로, 생성 샘플 필터링 강도(`loose`, `default`, `strict`)가 downstream anomaly detection 성능에 미치는 영향을 기록한다.

1단계 augmentation count 분석에서 `750`개는 F1, precision, AUPRC가 우수했고, `1000`개는 recall, F2, false negative 감소 측면에서 우수했다. 따라서 본 실험에서는 `augmentation_count = 750, 1000`만 사용한다.

## 1. 실험 목적

본 실험의 목적은 생성된 Masking Diffusion anomaly pool에서 샘플을 얼마나 엄격하게 선별해야 downstream classifier 성능이 좋아지는지 확인하는 것이다.

- `loose`: anomaly-like 샘플의 다양성을 더 많이 허용
- `default`: 기존 Research08에서 사용한 기본 filtering 기준
- `strict`: real anomaly와 더 가까운 샘플만 강하게 선별

핵심 질문은 다음과 같다.

> Filtering은 강할수록 좋은가, 아니면 anomaly 특성을 유지하면서 다양성을 확보하는 loose filtering이 더 효과적인가?

## 2. Filtering Strength 설정

| Strength | anomaly radius multiplier | normal radius multiplier | within anomaly range threshold | 해석 |
|---|---:|---:|---:|---|
| loose | 1.50 | 0.50 | 0.80 | 후보 샘플을 넓게 허용하여 다양성 확보 |
| default | 1.25 | 0.75 | 0.90 | 기존 Filtered Masking Diffusion 기본값 |
| strict | 1.00 | 1.00 | 0.95 | anomaly 중심에 가깝고 normal과 먼 샘플 위주로 제한 |

Filtering score는 기존 연구 흐름과 동일하게 다음 요소를 사용한다.

```text
quality_score = - dist_to_anomaly
                + 0.5 * dist_to_normal
                + 10.0 * within_anomaly_range
```

즉, real anomaly centroid와 가까우면서 normal centroid와 멀고, anomaly feature range 안에 많이 포함되는 generated sample을 우선 선택한다.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TOOLS_DIR = ROOT / "tools"
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

RESULT_DIR = ROOT / "data" / "research09" / "results"
RESULT_PATH = RESULT_DIR / "research09_filtering_strength.csv"
SUMMARY_PATH = RESULT_DIR / "research09_filtering_strength_summary.json"

ROOT, RESULT_PATH

## 3. 실험 실행

아래 셀은 `tools/research09_filtering_strength.py`의 `main()`을 호출하여 실험을 재현한다. 결과는 `data/research09/results/`에 저장된다.

실험 구성:

- generated pool: `data/research02/generated/diffusion_masked_windows.npz`
- normal train: 5,000 windows sampling
- real anomaly seed: 247 windows
- augmentation count: 750, 1000
- downstream model: `RandomForestClassifier`
- threshold selection: validation F1 기준

In [ ]:
# Re-run this cell when the experiment needs to be regenerated.
from research09_filtering_strength import main

main()

## 4. 결과 로드

In [ ]:
import json
import pandas as pd

results = pd.read_csv(RESULT_PATH)
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))

display_cols = [
    "augmentation_count",
    "filtering_strength",
    "threshold",
    "validation_f1",
    "precision",
    "recall",
    "f1",
    "f2",
    "auprc",
    "false_negative",
    "false_positive",
    "candidate_count",
    "fallback_used",
]

results[display_cols].sort_values(["augmentation_count", "filtering_strength"])

## 5. 핵심 결과

| Count | Filtering | Precision | Recall | F1 | F2 | AUPRC | FN | FP |
|---:|---|---:|---:|---:|---:|---:|---:|---:|
| 750 | loose | 0.8490 | 0.8547 | 0.8519 | 0.8536 | 0.9389 | 43 | 45 |
| 750 | default | 0.8911 | 0.7736 | 0.8282 | 0.7946 | 0.9276 | 67 | 28 |
| 750 | strict | 0.8817 | 0.7804 | 0.8280 | 0.7988 | 0.9070 | 65 | 31 |
| 1000 | loose | 0.9945 | 0.6149 | 0.7599 | 0.6657 | 0.9419 | 114 | 1 |
| 1000 | default | 0.9333 | 0.6622 | 0.7747 | 0.7030 | 0.9093 | 100 | 14 |
| 1000 | strict | 0.9038 | 0.7939 | 0.8453 | 0.8137 | 0.9326 | 61 | 25 |

In [ ]:
best_rows = []
for metric in ["precision", "recall", "f1", "f2", "auprc"]:
    row = results.loc[results[metric].idxmax()].copy()
    best_rows.append({
        "metric": metric,
        "best_count": int(row["augmentation_count"]),
        "best_filtering": row["filtering_strength"],
        "score": row[metric],
        "FN": int(row["false_negative"]),
        "FP": int(row["false_positive"]),
    })

pd.DataFrame(best_rows)

## 6. 시각화

In [ ]:
import matplotlib.pyplot as plt

plot_df = results.copy()
plot_df["condition"] = plot_df["augmentation_count"].astype(str) + " / " + plot_df["filtering_strength"]

metrics = ["precision", "recall", "f1", "f2", "auprc"]
ax = plot_df.set_index("condition")[metrics].plot(kind="bar", figsize=(11, 5), width=0.82)
ax.set_ylim(0.55, 1.02)
ax.set_xlabel("Augmentation count / filtering strength")
ax.set_ylabel("Final test score")
ax.legend(loc="lower right", ncol=3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 7. 해석

Filtering strength 분석 결과, `750 + loose` 조건이 전체적으로 가장 좋은 성능을 보였다. 이 조건은 F1, F2, recall이 가장 높고 false negative도 가장 적었다.

```text
750 + loose
F1     = 0.8519
F2     = 0.8536
Recall = 0.8547
FN     = 43
AUPRC  = 0.9389
```

`1000 + loose` 조건은 precision이 0.9945로 가장 높고 false positive가 1개로 가장 적었지만, recall이 0.6149로 낮고 false negative가 114개로 많았다. 따라서 제조 이상 탐지에서 결함 미탐을 줄이는 목적에는 적합하지 않다.

이 결과는 filtering을 무조건 엄격하게 하는 것이 최선이 아님을 보여준다. Filtered Masking Diffusion에서는 anomaly-like 특성을 유지하면서도 synthetic anomaly의 다양성을 확보하는 loose filtering이 recall-oriented 성능 개선에 더 유리했다.

## 8. 논문 서술 초안

생성 샘플 필터링 강도에 따른 성능 변화를 분석한 결과, 750개의 synthetic anomaly를 사용한 loose filtering 조건에서 가장 높은 F1-score와 F2-score가 나타났다. 이는 생성 샘플을 지나치게 엄격하게 제한하는 것보다, anomaly-like 특성을 유지하면서 일정 수준의 다양성을 허용하는 것이 downstream 이상 탐지 성능에 더 효과적임을 의미한다. 특히 loose filtering은 false negative를 가장 크게 감소시켜, 결함 미탐 비용이 큰 제조 이상 탐지 환경에서 유리한 설정으로 해석된다.